# Host the Stable Diffusion 1.5 LoRA demo on a free Colab T4

Select **Runtime > Change runtime type > T4 GPU** before running.

This notebook clones the repository, installs the training and app extras with uv, and
starts the Gradio demo with a public `--share` link. The adapter is loaded from Google
Drive on the first request.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU")
print(torch.cuda.get_device_name(0))

## Clone and install

Replace `YOUR_GITHUB_USER` with your GitHub username or paste the full repository URL.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/YOUR_GITHUB_USER/lora-style-imagegen.git"
REPO_DIR = Path("/content/lora-style-imagegen")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!python -m pip install -q uv
!uv sync --extra train --extra app --no-dev

## Mount Google Drive

The adapter path defaults to
`MyDrive/lora-style-imagegen/outputs/lora-style/lora`.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
ADAPTER_DIR = Path("/content/drive/MyDrive/lora-style-imagegen/outputs/lora-style/lora")
print(ADAPTER_DIR)
print((ADAPTER_DIR / "lora_metadata.json").exists())

## Start the demo

This cell runs until you stop it. The Gradio public URL is printed in the output. The
first generation downloads and loads the base model, so it may take several minutes.

In [ ]:
import subprocess

subprocess.run(
    [
        "uv",
        "run",
        "style-demo",
        "--adapter-dir",
        str(ADAPTER_DIR),
        "--share",
    ],
    check=True,
)

## Use the app

Open the `*.gradio.live` URL printed above. Enter a prompt containing the trigger token,
then adjust the seed, steps, guidance, and LoRA strength. Base and LoRA images use the
same seed; setting LoRA strength to 0 matches the base output. A seed of `-1` selects a
random seed and reports the value used.